In [20]:
import os
import numpy as np
import zarr
import tifffile as tiff
from pathlib import Path
from skimage.transform import resize

from instanseg import InstanSeg

# --- Fix InstanSeg WSI reader bug in this repo snapshot (read_slide needs TiffSlide) ---
from tiffslide import TiffSlide
import instanseg.inference_class as ic
ic.TiffSlide = TiffSlide

In [25]:
# ======================================================================================
# SETTINGS (edit per slide)
# ======================================================================================
ome_path = Path(r"/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/test/SLIDE-0329_crop_2048_segment_merge.ome.tif")
pixel_size_um = 0.325

out_dir = Path(r"/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/test")
out_dir.mkdir(parents=True, exist_ok=True)

# Which plane is which in the saved zarr (set once after a quick sanity check)
cells_plane  = 1
nuclei_plane = 0

# InstanSeg params (tune as needed)
inst = InstanSeg("fluorescence_nuclei_and_cells", verbosity=1)
inst_kwargs = dict(
    resolve_cell_and_nucleus=True,
    cleanup_fragments=True,
    seed_threshold=0.6,
    # mask_threshold=0.53,
    # peak_distance=5,
    # overlap_threshold=0.5,
    # window_size=64,
)

# Whole-slide tiling params
tile_size = 1000
overlap = 100

# Nimbus output location + naming
# Nimbus-Inference template expects something like:
#   segmentation/deepcell_output/<fov_name>_whole_cell.tiff
# Use fov_name derived from ome_path by default.
seg_out_dir = out_dir / "segmentation" / "deepcell_output"
seg_out_dir.mkdir(parents=True, exist_ok=True)
fov_name = ome_path.name.replace(".ome.tif", "").replace(".ome.tiff", "")

# Save cells (required for Nimbus); nuclei is optional but convenient
save_nuclei_too = True

Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/utils/../bioimageio_models/, loading
Requesting default device: cuda


In [22]:
# ======================================================================================
# 1) Run InstanSeg whole-slide (writes zarr next to ome_path)
# ======================================================================================
inst.eval_whole_slide_image(
    str(ome_path),
    pixel_size=pixel_size_um,   # override slide mpp if needed
    tile_size=tile_size,
    overlap=overlap,
    **inst_kwargs
)

Slide progress: 100%|█████████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.14it/s]


In [23]:
from pathlib import Path
import json
import sys
import traceback
REPO_ROOT = "/home/ratnayn/codex/mIF-pipeline/"

if "REPO_ROOT" not in globals():
    raise RuntimeError(
        "Set REPO_ROOT to the repository path before running this cell. Example: REPO_ROOT = '/abs/path/to/mIF-pipeline'"
    )

REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
if not (REPO_ROOT / "src").exists():
    raise RuntimeError(
        f"REPO_ROOT does not look like the repo root: {REPO_ROOT}. Expected to find {REPO_ROOT / 'src'}."
    )

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repo root: {REPO_ROOT}")
print(f"Source dir added to sys.path: {SRC_DIR}")


Repo root: /home/ratnayn/codex/mIF-pipeline
Source dir added to sys.path: /home/ratnayn/codex/mIF-pipeline/src


In [24]:
import zarr
import numpy as np
from pathlib import Path
from mif_pipeline import diagnose_label_overlap_instances

zarr_path = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/test/SLIDE-0329_crop_2048_segment_merge.ome_instanseg_prediction.zarr")  # or use run result
#zarr_path = Path("/mnt/e/SLIDE-0272_segment_merge.ome_instanseg_prediction.zarr/")  # or use run result
root = zarr.open(str(zarr_path), mode="r")

nuclear = np.asarray(root[0])
cells = np.asarray(root[1])

diagnose_label_overlap_instances(cells, nuclear)


{'overlap_pixels': 789367,
 'matching_pixels': 0,
 'mismatching_pixels': 789367,
 'exact_match': False,
 'example_mismatches': [{'y': 0, 'x': 84, 'cell_id': 2148, 'nuclear_id': 26},
  {'y': 0, 'x': 98, 'cell_id': 2123, 'nuclear_id': 1},
  {'y': 0, 'x': 99, 'cell_id': 2123, 'nuclear_id': 1},
  {'y': 0, 'x': 100, 'cell_id': 2123, 'nuclear_id': 1},
  {'y': 0, 'x': 101, 'cell_id': 2123, 'nuclear_id': 1}]}

In [3]:
# ======================================================================================
# 2) Locate output zarr and open it
# ======================================================================================
zarr_path = ome_path.parent / f"{ome_path.stem}{inst.prediction_tag}.zarr"
print("InstanSeg output zarr:", zarr_path)

root = zarr.open(str(zarr_path), mode="r")   # shape (2, h, w) at model resolution
print("zarr shape:", root.shape, "dtype:", root.dtype, "chunks:", root.chunks)

cells_z = root[cells_plane]      # (h, w)
nuc_z   = root[nuclei_plane]     # (h, w)

InstanSeg output zarr: /data1/lowes/ratnayn/Data/M10_admix/SLIDE-0272_segment_merge.ome_instanseg_prediction.zarr
zarr shape: (2, 39584, 38488) dtype: int32 chunks: (2, 2048, 2048)


In [4]:
# ======================================================================================
# 3) Determine target (H, W) from the *reference image canvas* (level 0)
#    This ensures the mask matches the input image dimensions for Nimbus.
# ======================================================================================
with tiff.TiffFile(str(ome_path)) as tf:
    # For OME-TIFF, pages[0] is typically full-res (level 0). For pyramids, this is still the base.
    page0 = tf.pages[0]
    H, W = int(page0.shape[-2]), int(page0.shape[-1])
print("Target full-res H,W:", (H, W))

Target full-res H,W: (60899, 59212)


In [6]:
# ======================================================================================
# 4) Load zarr planes into memory and upsample (nearest neighbor) to (H, W)
#    IMPORTANT: use order=0 for instance labels to preserve IDs.
# ======================================================================================
cells_lr = np.asarray(cells_z, dtype=np.uint32)
print("cells low-res:", cells_lr.shape, "max id:", int(cells_lr.max()))

cells_hr = resize(
    cells_lr, (H, W),
    order=0, preserve_range=True, anti_aliasing=False
).astype(np.uint32)
assert cells_hr.shape == (H, W)

if save_nuclei_too:
    nuc_lr = np.asarray(nuc_z, dtype=np.uint32)
    print("nuclei low-res:", nuc_lr.shape, "max id:", int(nuc_lr.max()))

    nuc_hr = resize(
        nuc_lr, (H, W),
        order=0, preserve_range=True, anti_aliasing=False
    ).astype(np.uint32)
    assert nuc_hr.shape == (H, W)

cells low-res: (39584, 38488) max id: 3275465
nuclei low-res: (39584, 38488) max id: 3275463


In [7]:
# ======================================================================================
# 5) Write outputs for Nimbus (uint32 instance masks)
#    Nimbus-Inference reads with imageio.imread + squeeze and casts to uint32.
#    OME metadata is not required; plain tiled TIFF is fine.
# ======================================================================================
cells_out = seg_out_dir / f"{fov_name}_whole_cell.ome.tif"
tiff.imwrite(
    str(cells_out),
    cells_hr,
    dtype=np.uint32,
    bigtiff=True,
    compression="zlib",
    tile=(256, 256),
)
print("Wrote whole-cell mask for Nimbus:", cells_out)

if save_nuclei_too:
    nuc_out = seg_out_dir / f"{fov_name}_nuclear.ome.tif"
    tiff.imwrite(
        str(nuc_out),
        nuc_hr,
        dtype=np.uint32,
        bigtiff=True,
        compression="zlib",
        tile=(256, 256),
    )
    print("Wrote nuclei mask (optional):", nuc_out)

Wrote whole-cell mask for Nimbus: /data1/lowes/ratnayn/Data/M10_admix/SLIDE-0272_segmentation/segmentation/deepcell_output/SLIDE-0272_segment_merge_whole_cell.tiff
Wrote nuclei mask (optional): /data1/lowes/ratnayn/Data/M10_admix/SLIDE-0272_segmentation/segmentation/deepcell_output/SLIDE-0272_segment_merge_nuclear.tiff
